# Generate Reranking Matrices with Llama Models on Colab

This notebook generates pairwise comparison matrices for BEIR datasets using Llama models.
The matrices can be used with the IReranker evaluation framework.

**Requirements:**
- Google Colab with GPU runtime (V100, A100, or T4)
- Google Drive for storing outputs and checkpoints
- HuggingFace account for accessing Llama models

**Runtime Setup:**
1. Go to Runtime > Change runtime type
2. Select GPU as hardware accelerator
3. Choose High-RAM if available

**Generated matrices are compatible with existing IReranker code.**

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create output directories
!mkdir -p /content/drive/MyDrive/IReranker/matrices
!mkdir -p /content/drive/MyDrive/IReranker/checkpoints

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf

In [ ]:
# Clone IReranker repository
import os
if not os.path.exists('/content/IReranker'):
    !git clone https://github.com/YOUR_USERNAME/IReranker.git /content/IReranker
else:
    print("Repository already cloned")

# Install IReranker
%cd /content/IReranker
!pip install -q -e .

In [ ]:
# Login to HuggingFace (required for Llama models)
from huggingface_hub import notebook_login
notebook_login()

## 2. Configuration

In [ ]:
# Model configuration
MODELS = {
    "llama-8b": "meta-llama/Llama-3.1-8B-Instruct",
    "llama-70b": "meta-llama/Llama-3.1-70B-Instruct",
    # Note: 405B model requires multiple GPUs or heavy quantization
    # "llama-405b": "meta-llama/Llama-3.1-405B-Instruct",
}

# Dataset configuration
DATASETS = [
    "scifact",      # Small dataset for testing (300 queries, 5K docs)
    "dl-2019",      # TREC Deep Learning 2019
    "dl-2020",      # TREC Deep Learning 2020
    # Add more datasets as needed:
    # "trec-covid", "nfcorpus", "fiqa", "webis-touche2020"
]

# Quantization settings (adjust based on GPU memory)
QUANTIZATION = {
    "llama-8b": "8bit",    # 8B fits in 16GB with 8-bit
    "llama-70b": "4bit",   # 70B requires 4-bit for V100
}

# Generation settings
MAX_QUERIES = None  # Set to small number (e.g., 10) for testing, None for full dataset
CHECKPOINT_INTERVAL = 500  # Save checkpoint every N comparisons

# Output paths
OUTPUT_DIR = "/content/drive/MyDrive/IReranker/matrices"
CHECKPOINT_DIR = "/content/drive/MyDrive/IReranker/checkpoints"

print("Configuration loaded:")
print(f"  Models: {list(MODELS.keys())}")
print(f"  Datasets: {DATASETS}")
print(f"  Max queries: {MAX_QUERIES or 'All'}")
print(f"  Output: {OUTPUT_DIR}")

## 3. Download BEIR Datasets

In [ ]:
# Download BEIR datasets (this will be cached)
from beir import util
import os

BEIR_DIR = "/content/IReranker/data/external/beir"
os.makedirs(BEIR_DIR, exist_ok=True)

for dataset in DATASETS:
    dataset_path = os.path.join(BEIR_DIR, dataset)
    if os.path.exists(dataset_path):
        print(f"✓ {dataset} already downloaded")
    else:
        print(f"Downloading {dataset}...")
        url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
        util.download_and_unzip(url, BEIR_DIR)
        print(f"✓ {dataset} downloaded")

print("\nAll datasets ready!")

## 4. Generate Matrices

In [ ]:
from ireranker.oracles import TransformerOracle
from ireranker.data.loaders import load_beir_dataset
from pathlib import Path
import time

# Track progress
results = []

for model_key, model_path in MODELS.items():
    print(f"\n{'='*80}")
    print(f"Model: {model_key} ({model_path})")
    print(f"{'='*80}\n")
    
    # Initialize oracle
    quantization = QUANTIZATION.get(model_key)
    oracle = TransformerOracle(
        model_name=model_path,
        device="cuda",
        quantization=quantization,
    )
    
    # Enable checkpoints
    checkpoint_dir = Path(CHECKPOINT_DIR) / model_key
    oracle.enable_checkpoints(checkpoint_dir, interval=CHECKPOINT_INTERVAL)
    
    for dataset in DATASETS:
        print(f"\nProcessing dataset: {dataset}")
        start_time = time.time()
        
        try:
            # Load dataset into oracle
            oracle.load_dataset(dataset, split="test")
            
            # Load ranking tasks to trigger comparisons
            # We'll simulate the evaluation process
            ranking_dataset = load_beir_dataset(
                dataset,
                split="test",
                max_queries=MAX_QUERIES,
                matrix_model=None  # We're generating, not loading
            )
            
            # Process each task to generate comparisons
            total_comparisons = 0
            for task_idx, task in enumerate(ranking_dataset.tasks):
                oracle.set_task(task)
                oracle.reset_comparisons()
                
                # Generate all pairwise comparisons for this task
                n_candidates = len(task.candidate_ids)
                for i in range(n_candidates):
                    for j in range(i + 1, n_candidates):
                        # This will trigger model inference and cache the result
                        _ = oracle.lt(i, j)
                
                total_comparisons += oracle.comparisons
                
                if (task_idx + 1) % 10 == 0:
                    elapsed = time.time() - start_time
                    print(f"  Progress: {task_idx + 1}/{len(ranking_dataset.tasks)} tasks, "
                          f"{total_comparisons} comparisons, {elapsed:.1f}s")
            
            # Save final matrix
            output_path = Path(OUTPUT_DIR) / model_key / f"{dataset}.pkl"
            oracle.save_matrix(output_path)
            
            elapsed = time.time() - start_time
            result = {
                "model": model_key,
                "dataset": dataset,
                "queries": len(ranking_dataset.tasks),
                "comparisons": total_comparisons,
                "time_seconds": elapsed,
                "status": "success"
            }
            results.append(result)
            
            print(f"\n✓ {dataset} completed:")
            print(f"  - Queries: {len(ranking_dataset.tasks)}")
            print(f"  - Comparisons: {total_comparisons}")
            print(f"  - Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
            print(f"  - Output: {output_path}")
            
        except Exception as e:
            print(f"\n✗ Error processing {dataset}: {e}")
            import traceback
            traceback.print_exc()
            
            result = {
                "model": model_key,
                "dataset": dataset,
                "status": "error",
                "error": str(e)
            }
            results.append(result)
            continue

print("\n" + "="*80)
print("Generation Complete!")
print("="*80)

## 5. Summary and Results

In [ ]:
# Display summary
import pandas as pd

df = pd.DataFrame(results)
print("\nGeneration Summary:")
print(df.to_string(index=False))

# Calculate statistics
successful = df[df['status'] == 'success']
if len(successful) > 0:
    print(f"\nStatistics:")
    print(f"  Total matrices generated: {len(successful)}")
    print(f"  Total comparisons: {successful['comparisons'].sum():,}")
    print(f"  Total time: {successful['time_seconds'].sum()/3600:.2f} hours")
    print(f"  Average comparisons per query: {successful['comparisons'].sum() / successful['queries'].sum():.0f}")

failed = df[df['status'] == 'error']
if len(failed) > 0:
    print(f"\nFailed: {len(failed)}")
    print(failed[['model', 'dataset', 'error']].to_string(index=False))

## 6. Download Matrices (Optional)

In [ ]:
# Zip matrices for easy download
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"llama_matrices_{timestamp}"
zip_path = f"/content/drive/MyDrive/IReranker/{zip_name}"

print(f"Creating archive: {zip_path}.zip")
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"✓ Archive created: {zip_path}.zip")
print(f"\nYou can download it from Google Drive at: MyDrive/IReranker/{zip_name}.zip")

## 7. Next Steps

After generating matrices:

1. **Download matrices** from Google Drive
2. **Place in IReranker**: Copy to `data/external/reranking-matrices/llama/`
3. **Run evaluation**:
   ```bash
   make beir-eval ARGS="--matrix-models llama-8b"
   ```
4. **Compare results**: Check `reports/beir-metrics/`

**Matrix Format:**
The generated PKL files are compatible with all existing IReranker oracles and rankers.
Each file contains a dictionary with keys `(query_id, doc_a, doc_b)` and comparison results.

**Checkpoints:**
If generation is interrupted, you can resume by loading the latest checkpoint:
```python
oracle.load_matrix('/content/drive/MyDrive/IReranker/checkpoints/llama-8b/scifact_checkpoint_5000.pkl')
```